# Parallax v0.3.0 Validation Walkthrough

This notebook provides an interactive, step-by-step walkthrough of the Parallax reasoning framework. You will see exactly how we load data, detect communities (Attention Heads), build holographic indices, and perform multi-hop reasoning with Community-Structured Attention (CSA).

## Objectives:
1. **Visualize the Graph**: See the topological structure of the canonical `toy_graph`.
2. **DSCF in Action**: Observe how Local (LPA) and Global (Modularity) signals fuse to form communities.
3. **Holographic Indexing**: Inspect how we compress community knowledge into privacy-preserving signatures.
4. **Reasoning Journey**: Follow a multi-hop traversal and see the attention scores at every step.
5. **Resource Governance**: See how the `ResourceGovernor` protects the system from infinite reasoning loops.
6. **The Bridge Bonus (EF-005)**: Demonstration of how we solve the 'Type Alignment Trap'.

In [ ]:
import os
import sys
import json
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np

# Add project root to path
sys.path.insert(0, str(Path(os.getcwd()).parent))

from adapters.csv_adapter import load_csv_adapter
from core.community_engine import dscf_communities, modularity_score
from core.embedding_engine import RandomEngine
from core.attention_engine import CSAEngine
from core.structural_encoder import build_community_distance_matrix, adjacent_community_pairs
from core.holographic_index import build_signatures
from core.resource_governor import ResourceGovernor
from reasoning.traversal import BeamTraversal
from reasoning.answer_extractor import extract

print("Parallax Framework Ready.")

## 1. Load the Graph Environment

We use the `toy_graph.csv` which contains two primary conceptual clusters: **History** (Roman/Greek) and **Science** (Modern Physics). We use the `NetworkXAdapter` (via `load_csv_adapter`) to bridge the CSV data to our reasoning engines.

In [ ]:
csv_path = Path("../tests/fixtures/toy_graph.csv")
adapter = load_csv_adapter(str(csv_path))
G = adapter.to_networkx()

print(f"Loaded graph with {G.number_of_nodes()} nodes and {G.number_of_edges()} edges.")

plt.figure(figsize=(10, 8))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, with_labels=True, node_color='lightblue', edge_color='gray', node_size=1500, font_size=8)
plt.title("Canonical Toy Graph Topology")
plt.show()

## 2. DSCF: Forming Attention Heads

We now run the **Dual-Signal Community Fusion** algorithm. DSCF identifies conceptual clusters that act as "attention heads" during reasoning. Notice how it correctly separates the scientists from the historical figures.

In [ ]:
# Run DSCF
np.random.seed(42)
parts = dscf_communities(G, resolution=1.0)
q = modularity_score(G, parts)
print(f"Detected {len(parts)} communities with Modularity Q = {q:.4f}")

# Map nodes to colors based on community
community_map = {node: i for i, members in enumerate(parts) for node in members}
adapter.community_map = community_map # Attach to adapter for engine lookups
colors = [community_map[node] for node in G.nodes()]

plt.figure(figsize=(10, 8))
nx.draw(G, pos, with_labels=True, node_color=colors, cmap=plt.cm.Set3, node_size=1500, font_size=8)
plt.title("Visualizing Attention Heads (DSCF Communities)")
plt.show()

## 3. Holographic Indexing

Parallax v0.3.0 uses **Holographic Signatures** to enable federated reasoning without sharing raw data. We compress each community into a signature containing a **Centroid Embedding** and a **Bloom Filter** of its entities.

In [ ]:
# Setup Embeddings for signatures
embed_engine = RandomEngine(dim=64)
embeddings = embed_engine.encode_entities({n: n for n in G.nodes()})
adapter.embeddings = embeddings

signatures = build_signatures(adapter, community_map, embeddings)
print(f"Generated {len(signatures)} holographic signatures.")

# Inspect a signature
sig = signatures[0]
print(f"\nCommunity {sig.community_id} Signature Sample:")
print(f"- Size: {sig.size} entities")
print(f"- Centroid (first 5): {sig.centroid[:5]}")
print(f"- Bloom Filter: {sig.bloom.to_hex()[:50]}...")

## 4. The Reasoning Journey

We will now ask Parallax: **"What is connected to 'newton'?"**

We'll use the **Metaedge Bridge Bonus (EF-005)** to favor the `INFLUENCED` relationship, simulating a user interested in intellectual lineage.

In [ ]:
# 1. Setup CSA Engine with structural metadata
dist_matrix = build_community_distance_matrix(G, community_map)
adj_pairs = adjacent_community_pairs(G, community_map)

edge_weights = {"INFLUENCED": 0.4}
csa_engine = CSAEngine(
    adapter=adapter, 
    edge_type_weights=edge_weights
)
csa_engine.set_community_graph(dist_matrix, adj_pairs)

# 2. Configure Beam Traversal
traversal = BeamTraversal(
    adapter=adapter,
    csa_engine=csa_engine,
    beam_width=5,
    max_hop=2,
    max_budget=100
)

# 3. Run Reasoner
paths = traversal.traverse(["newton"])
answers = extract(paths, top_k=5)

print(f"Total expansions consumed: {traversal.expansions}/{traversal.max_budget}")
print("\nTop reasoning paths found:")
for i, ans in enumerate(answers):
    print(f"\n[{i+1}] {ans.entity_id} (Score: {ans.score:.4f})")
    print(f"    Path: {' -> '.join(ans.best_path.nodes)}")
    print(f"    Heads: {ans.community_trace}")
    print(f"    Breakdown: {json.dumps(ans.score_breakdown, indent=4)}")

## 5. Deep Dive: Attention Weights & Penalties

Let's look under the hood at how the **CSA Formula** scores edges. We'll compare an internal intellectual link (`newton -> einstein`) to a cross-domain jump (`newton -> caesar`).

The scores below show how **Semantic Similarity**, **Community Coherence**, and **Edge Relevance** combine.

In [ ]:
def inspect_edge(u, v, rel):
    # Manually calculate scores from csa_engine components
    w = csa_engine.compute_weight(u, v, hop=1, edge_type=rel)
    c_score = csa_engine.community_score(u, v)
    e_bonus = csa_engine.edge_type_weights.get(rel, 0.0)
    
    print(f"Edge: {u} --[{rel}]--> {v}")
    print(f"- Total Attention Score: {w:.4f}")
    print(f"- Community Coherence:   {c_score:.4f} (Beta Signal)")
    print(f"- Edge-Type Bonus:       {e_bonus:.4f} (Gamma Signal)")
    print("-" * 30)

inspect_edge("newton", "einstein", "INFLUENCED")
inspect_edge("newton", "caesar", "NONE") # Simulating a jump

## 6. Resource Governance & Safety

In large KGs, reasoning can spiral out of control. Parallax uses a `ResourceGovernor` to protect the process and system health.

In [ ]:
governor = ResourceGovernor()
stats = governor.get_current_stats()
print("Current System Health:")
print(f"- System RAM: {stats['system_ram_pct']}% used")
print(f"- Process RSS: {stats['process_rss_mb']} MB")

max_budget = 10
print(f"\nSimulating expansion with budget {max_budget}:")
for i in range(1, 15):
    allowed = governor.can_expand(i, max_budget)
    status = "OK" if allowed else "HALT"
    print(f"Step {i}: {status}")
    if not allowed: break

## Conclusion

This walkthrough demonstrates that Parallax v0.3.0 is a production-hardened reasoning engine. It doesn't just 'find' data—it **reasons** through it using structural attention, protects itself with resource governance, and enables secure collaboration through holographic indexing.